## date of birth

In [1]:
import pandas as pd
import datetime
import re
import json

c:\Users\sumse\miniconda3\envs\data_quality\lib\site-packages\numpy\_distributor_init.py:30: UserWarning: loaded more than 1 DLL from .libs:
c:\Users\sumse\miniconda3\envs\data_quality\lib\site-packages\numpy\.libs\libopenblas_v0.3.21-gcc_8_3_0.dll
c:\Users\sumse\miniconda3\envs\data_quality\lib\site-packages\numpy\.libs\libopenblas_v0.3.23-gcc_8_3_0.dll
  warnings.warn("loaded more than 1 DLL from .libs:"


In [10]:
# import data

date_of_birth = pd.read_csv(r"D:\Portfolio\Data Engineer Intern at BTS Group (Jul - Nov 2023)\Data Quality Improvement Project\notebook_python\date_of_birth\1_customer_comp_dateofbirth.csv" , encoding = 'utf8')

date_of_birth.shape

(115, 8)

In [3]:
# import dictionary of education(json)

with open(r"D:\Portfolio\Data Engineer Intern at BTS Group (Jul - Nov 2023)\Data Quality Improvement Project\notebook_python\date_of_birth\date_of_birth_pattern.json",encoding = 'utf-8') as fin :
    pattern_dict = json.load(fin)

In [7]:
pd.set_option('display.max_rows', None)

In [8]:
date_of_birth.head(5)

,subject,no,id,case,detail case,flag,date_of_birth,patten
0,date_of_birth,1,date_of_birth-USA-1,USA,normal,good,06-01-1986,English(United States)
1,date_of_birth,2,date_of_birth-USA-2,USA,normal,good,06/11/1986,English(United States)
2,date_of_birth,3,date_of_birth-USA-3,USA,normal,good,June-23-1986,English(United States)
3,date_of_birth,4,date_of_birth-USA-4,USA,normal,good,Jun-23-1986,English(United States)
4,date_of_birth,5,date_of_birth-USA-5,USA,normal,good,"June 20, 1986",English(United States)


In [9]:
date_of_birth.tail(5)

,subject,no,id,case,detail case,flag,date_of_birth,patten
110,date_of_birth,111,date_of_birth-THA-111,THA,other,bad,๑๖-มีนามคม-A๕๒๕,Thai
111,date_of_birth,112,date_of_birth-THA-112,THA,other,bad,๒๔@มี.ค.A๒๕,Thai
112,date_of_birth,113,date_of_birth-THA-113,THA,other,bad,12-กค-๒๕๒๕,Thai
113,date_of_birth,114,date_of_birth-THA-114,THA,other,bad,29-กพ-๒๕๒๕,Thai
114,date_of_birth,115,date_of_birth-THA-115,THA,other,bad,๑๓:๓๐ น. ๑๔-๐๒-๒๕๒๕,Thai


In [11]:
# lower case 
# remove special char / days
# convert thai num to arabic num
def standard_transform(text_input):
    text_input = text_input.lower().replace('_' , ' ').replace('.' , '').replace('-' , ' ').replace(' น' , ' ')
    text_input = re.sub(r'[^ก-๛a-z\w\s:]', ' ', text_input)
    
    days = ["monday","tuesday","wednesday","thursday","friday","saturday","sunday","mon","tue","wed","thu","fri","sat","sun",
            "วันจันทร์","วันอังคาร","วันพุธ","วันพฤหัสบดี","วันศุกร์","วันเสาร์","วันอาทิตย์","จ", "อ", "พ", "พฤ", "ศ", "ส", "อา"]

    for day in days :
        if day in text_input :
            text_input = text_input.replace(day , '')
            return text_input
    
    thai_digits = '๐๑๒๓๔๕๖๗๘๙'
    western_digits = '0123456789'
    text_input = text_input.translate(str.maketrans(thai_digits, western_digits))
    
    return text_input



# create column for transform data

date_of_birth['date_of_birth_transform'] = date_of_birth['date_of_birth'].apply(lambda date_of_birth : standard_transform(date_of_birth))

In [12]:
# create time column
def take_time(text_input) :
    try :
        result = text_input.lower()
        result = re.search(r'([0-9]{,2}:[0-9]{,2}:[0-9]{,2}|[0-9]{,2}:[0-9]{,2}\s?[pm|am]{,2})' , text_input).group()
        return result
    except AttributeError :
        return '-'
    


# create column for time
date_of_birth['time'] = date_of_birth['date_of_birth_transform'].apply(lambda date_of_birth : take_time(date_of_birth))

# create column for date : if the data be in time , not be in date
date_of_birth['date'] = date_of_birth.apply(lambda date_of_birth : date_of_birth['date_of_birth_transform'].replace(date_of_birth['time'] , '') , axis = 1)

In [13]:
# convert char month to num month
def convert_month(text_input) :
    for pattern,value in pattern_dict.items():
        if re.search(pattern , text_input):
            result = re.sub(pattern , ' '+value+' ' , text_input)
            return result
        else:
            continue
    return text_input



# create column for convert char month(Jan,...,Dec) to num month(01 - 12)
date_of_birth['date'] = date_of_birth['date'].apply(lambda date : convert_month(date))

In [14]:
# remove text in date            
def remove_text(text_input) :
    if re.search(r'[ก-๛]|[a-z]' , text_input) :
        return None
    else :
        return text_input
    


# remove text in column date
date_of_birth['date'] = date_of_birth['date'].apply(lambda date : remove_text(date))

In [15]:
# tranform other format to d-m-y
def transform_date_format(text_input):
    if text_input is None:
        return None
    
    formats = ['%m %d %Y', '%Y %d %m', '%d %m %Y', '%Y %m %d', '%m %d %y', '%y %d %m', '%d %m %y', '%y %m %d']
    
    for date_format in formats :
        try:
            text_input = text_input.strip()
            date_obj = datetime.datetime.strptime(text_input, date_format)
            result = date_obj.strftime('%d-%m-%Y')
            return result
        except ValueError:
            pass
    
    return None




# transform format
date_of_birth['standard_date_of_birth'] = date_of_birth['date'].apply(lambda date :transform_date_format(date))

In [16]:
# conver year era to christian year era
def convert_year_era(text_input):
    try:
        split = text_input.split('-')
        
        if len(split) != 3:
            return None
        
        day, month, year_bs = split
        
        current_date = datetime.date.today()
        current_year_ad = current_date.year
        current_year_bs = current_year_ad + 543 - 200
        
        if len(year_bs) == 4 and int(year_bs) >= current_year_bs:
            christian_year = int(year_bs) - 543
            return f"{day}-{month}-{christian_year}"
        else:
            return f"{day}-{month}-{year_bs}"
    except Exception as e:
        return None

# convert buddhist year era to christian year era
date_of_birth['standard_date_of_birth'] = date_of_birth['standard_date_of_birth'].apply(lambda standard_date_of_birth : convert_year_era(standard_date_of_birth))

In [17]:
# calculate age
def check_age(text_input):
    try:
        split = text_input.split('-')
        
        for year in split:
            if len(year) == 4:
                birth_year = int(year)
                current_year = datetime.datetime.now().year
                
                if birth_year > current_year:
                    return None
                else:
                    age = current_year - birth_year
                    if age >= 0 :
                        return str(int(age))
                    else :
                        return None
            elif text_input == None :
                return None

    except:
        return None
    


# create column for age
date_of_birth['age'] = date_of_birth['standard_date_of_birth'].apply(lambda standard_date_of_birth : check_age(standard_date_of_birth))

In [18]:
# validate year of birth < current year
def check_year(text_input) :
    try:
        split = text_input.split('-')
        
        for year in split:
            if len(year) == 4:
                birth_year = int(year)
                current_year = datetime.datetime.now().year

                if birth_year < current_year :
                    return True
                else :
                    return False
    except:
        return None
    


# create column for validate year of birth < current year
date_of_birth['validate_year_of_birth'] = date_of_birth['standard_date_of_birth'].apply(lambda standard_date_of_birth : check_year(standard_date_of_birth))

In [19]:
# validate age < 100
def age_less_100(text_input) :
    try :
        if int(text_input) < 100 :
            return True
        else :
            return False
    except :
        return None
    



# create column for validate age < 100
date_of_birth['validate_age'] = date_of_birth['age'].apply(lambda age : age_less_100(age))

In [20]:
def validate_standard_format(text_input) :
        if isinstance(text_input, str) and re.search(r'^[0-9]{2}\-[0-9]{2}\-[0-9]{4}$' , text_input) :
            return True
        else :
            return False

    


# create column for validate standard format
date_of_birth['validate_standard_format'] = date_of_birth['standard_date_of_birth'].apply(lambda standard_date_of_birth : validate_standard_format(standard_date_of_birth))

In [21]:
# last result
def result(text_input, standard):
    if isinstance(text_input , str) and standard == None :
        return text_input , 0
    
    elif text_input == standard :
        return text_input , 1
    
    elif text_input != standard :
        return standard , 2
    
    elif text_input == None and standard == None :
        return text_input , 3
    

date_of_birth[['std_output' , 'status']] = date_of_birth.apply(lambda row: pd.Series(result(row['date_of_birth'], row['standard_date_of_birth'])), axis=1)

In [22]:
selected_cols = ['id','date_of_birth','standard_date_of_birth', 'validate_standard_format', 'age','validate_year_of_birth','validate_age','std_output' ,'status']

date_of_birth = date_of_birth[selected_cols]

In [23]:
date_of_birth.tail(5)

,id,date_of_birth,standard_date_of_birth,validate_standard_format,age,validate_year_of_birth,validate_age,std_output,status
110,date_of_birth-THA-111,๑๖-มีนามคม-A๕๒๕,None,False,None,None,None,๑๖-มีนามคม-A๕๒๕,0
111,date_of_birth-THA-112,๒๔@มี.ค.A๒๕,None,False,None,None,None,๒๔@มี.ค.A๒๕,0
112,date_of_birth-THA-113,12-กค-๒๕๒๕,07-12-1982,True,44,True,True,07-12-1982,2
113,date_of_birth-THA-114,29-กพ-๒๕๒๕,None,False,None,None,None,29-กพ-๒๕๒๕,0
114,date_of_birth-THA-115,๑๓:๓๐ น. ๑๔-๐๒-๒๕๒๕,14-02-1982,True,44,True,True,14-02-1982,2
